# Aurora Siger — Sistema de Análise de Telemetria

Sistema de verificação pré-lançamento com análise por IA (Google Gemini).

---
**Instruções:**
1. Execute a célula de instalação (Célula 1)
2. Insira sua chave da API Gemini (Célula 2)
3. Preencha os dados de telemetria (Célula 3)
4. Execute a análise (Célula 4)

In [ ]:
# Célula 1 — Instalação de dependências
!pip install -q google-generativeai ipywidgets


In [ ]:
# Célula 2 — Configuração da API Key
import os
import getpass
import google.generativeai as genai

# Tenta ler do Colab Secrets, caso contrário pede digitação
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('API Key carregada via Colab Secrets.')
except Exception:
    GEMINI_API_KEY = getpass.getpass('Insira sua Gemini API Key: ')

genai.configure(api_key=GEMINI_API_KEY)
modelo = genai.GenerativeModel('gemini-2.5-flash')
print('Modelo Gemini configurado.')


In [ ]:
# Célula 3 — Entrada de dados via widgets
import ipywidgets as widgets
from IPython.display import display

style  = {'description_width': '260px'}
layout = widgets.Layout(width='500px')

w_temp_interna      = widgets.FloatSlider(value=22.0, min=-10, max=40,  step=0.5, description='Temperatura interna (C):', style=style, layout=layout)
w_temp_externa      = widgets.FloatSlider(value=20.0, min=-5,  max=45,  step=0.5, description='Temperatura externa (C):', style=style, layout=layout)
w_nivel_energia     = widgets.FloatSlider(value=85.0, min=0,   max=100, step=1.0, description='Nivel de energia (%):', style=style, layout=layout)
w_nivel_combustivel = widgets.FloatSlider(value=90.0, min=0,   max=100, step=1.0, description='Combustivel RP-1 (%):', style=style, layout=layout)
w_nivel_oxidante    = widgets.FloatSlider(value=90.0, min=0,   max=100, step=1.0, description='Oxidante LOX (%):', style=style, layout=layout)
w_pressao_tanque    = widgets.FloatSlider(value=3.5,  min=0,   max=6,   step=0.1, description='Pressao do tanque (bar):', style=style, layout=layout)
w_integridade       = widgets.ToggleButtons(value='OK', options=['OK', 'ERRO'], description='Integridade estrutural:', style=style)

print('=' * 50)
print('  AURORA SIGER — Dados de Telemetria')
print('=' * 50)
display(w_temp_interna, w_temp_externa, w_nivel_energia,
        w_nivel_combustivel, w_nivel_oxidante, w_pressao_tanque, w_integridade)


In [ ]:
# Célula 4 — Verificações e análise IA

# Leitura dos widgets
temp_interna      = w_temp_interna.value
temp_externa      = w_temp_externa.value
nivel_energia     = w_nivel_energia.value
nivel_combustivel = w_nivel_combustivel.value
nivel_oxidante    = w_nivel_oxidante.value
pressao_tanque    = w_pressao_tanque.value
integridade       = 1 if w_integridade.value == 'OK' else 0

gradiente_termico = abs(temp_interna - temp_externa)

# Verificações
checks = [
    ('Temperatura interna',       -10 <= temp_interna <= 40,      'Fora do limite ({}C). Esperado: -10C a 40C.'.format(temp_interna)),
    ('Temperatura externa',       -5  <= temp_externa <= 45,      'Fora do limite ({}C). Esperado: -5C a 45C.'.format(temp_externa)),
    ('Gradiente termico',         gradiente_termico <= 40,         'Gradiente excessivo ({:.1f}C). Max: 40C.'.format(gradiente_termico)),
    ('Nivel de energia',          nivel_energia >= 70,             'Energia insuficiente ({:.1f}%). Min: 70%.'.format(nivel_energia)),
    ('Combustivel RP-1',          nivel_combustivel >= 80,         'Combustivel insuficiente ({:.1f}%). Min: 80%.'.format(nivel_combustivel)),
    ('Oxidante LOX',              nivel_oxidante >= 80,            'Oxidante insuficiente ({:.1f}%). Min: 80%.'.format(nivel_oxidante)),
    ('Pressao do tanque',         2.5 <= pressao_tanque <= 4.5,   'Pressao fora do intervalo ({:.2f} bar). Esperado: 2.5-4.5 bar.'.format(pressao_tanque)),
    ('Integridade estrutural',    integridade == 1,                'Falha estrutural detectada.'),
]

print('\n' + '=' * 50)
print('  AURORA SIGER — Verificacao de Sistemas')
print('=' * 50)

todas_ok = True
for nome, passou, erro in checks:
    if passou:
        print('  >> {}: OK'.format(nome))
    else:
        print('  >> {}: FALHA'.format(nome))
        print('     {}'.format(erro))
        todas_ok = False

print('\n' + '=' * 50)
if todas_ok:
    print('\n✅ PRONTO PARA DECOLAR')
else:
    print('\n❌ DECOLAGEM ABORTADA')
print('=' * 50)

# Analise Gemini
status_texto = 'PRONTO PARA DECOLAR' if todas_ok else 'DECOLAGEM ABORTADA'
integridade_texto = 'OK' if integridade == 1 else 'ERRO'

prompt = (
    'Voce e um sistema de analise de telemetria aeroespacial. Analise os dados da Missao Aurora Siger:\n\n'
    'TELEMETRIA:\n'
    '- Temperatura interna: {}C\n'.format(temp_interna) +
    '- Temperatura externa: {}C\n'.format(temp_externa) +
    '- Gradiente termico: {:.1f}C\n'.format(gradiente_termico) +
    '- Nivel de energia: {:.1f}%\n'.format(nivel_energia) +
    '- Nivel de combustivel (RP-1): {:.1f}%\n'.format(nivel_combustivel) +
    '- Nivel de oxidante (LOX): {:.1f}%\n'.format(nivel_oxidante) +
    '- Pressao do tanque: {:.2f} bar\n'.format(pressao_tanque) +
    '- Integridade estrutural: {}\n'.format(integridade_texto) +
    '- Status geral: {}\n\n'.format(status_texto) +
    'Responda em portugues com:\n'
    '1. Classificacao de cada dado (normal / atencao / critico)\n'
    '2. Possiveis anomalias identificadas\n'
    '3. Sugestoes de risco\n'
    '4. Formato de resposta: Texto simples, direto e claro. Sem formatacao adicional.'
)

print('\n[ Analise IA — Gemini ] Aguardando resultado...\n')
resposta = modelo.generate_content(prompt)
print(resposta.text)
